In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATAPATH = Path("/Users/sebastianodutola/Projects/Carbon_Intensity/Data/backtest_data.py")
df = pd.read_parquet(DATAPATH)
# drop duplicates that might have wiggled through the net
df = df.drop_duplicates(subset="from")
df["from"] = df["from"].apply(np.datetime64)
df = df.reset_index(drop=True)

# Check for timestamp irregularities in the data
df_check = df.copy()
ts = df_check["from"].to_numpy().astype(np.datetime64)
dts = (ts[1:] - ts[:-1]).astype(dtype="timedelta64[s]")
dts = np.concat([dts, [np.timedelta64(30,'m')]])
df_check["dts"] = dts
idx = df_check[df_check["dts"].astype(int)!=1800].index
idxl = []
for i in idx:
    idxl.extend([i-1,i,i+1])
df_check.loc[idxl]

/Users/sebastianodutola/.virtualenvs/datascience/lib/python3.11/site-packages/pandas/core/algorithms.py:1743: UserWarning: no explicit representation of timezones available for np.datetime64
  return lib.map_infer(values, mapper, convert=convert)


,from,to,intensity,dts
62481,2021-04-19 16:00:00,2021-04-19T16:30Z,"{'actual': 273.0, 'forecast': 260.0, 'index': ...",0 days 00:30:00
62482,2021-04-19 16:30:00,2021-04-19T17:00Z,"{'actual': 278.0, 'forecast': 270.0, 'index': ...",0 days 01:00:00
62483,2021-04-19 17:30:00,2021-04-19T18:00Z,"{'actual': 286.0, 'forecast': 274.0, 'index': ...",0 days 00:30:00
62490,2021-04-19 21:00:00,2021-04-19T21:30Z,"{'actual': 284.0, 'forecast': 274.0, 'index': ...",0 days 00:30:00
62491,2021-04-19 21:30:00,2021-04-19T22:00Z,"{'actual': 284.0, 'forecast': 286.0, 'index': ...",0 days 01:30:00
62492,2021-04-19 23:00:00,2021-04-19T23:30Z,"{'actual': 281.0, 'forecast': 274.0, 'index': ...",0 days 00:30:00
74522,2021-12-26 14:00:00,2021-12-26T14:30Z,"{'actual': None, 'forecast': 155.0, 'index': '...",0 days 00:30:00
74523,2021-12-26 14:30:00,2021-12-26T15:00Z,"{'actual': None, 'forecast': 158.0, 'index': '...",1 days 03:30:00
74524,2021-12-27 18:00:00,2021-12-27T18:30Z,"{'actual': 179.0, 'forecast': 182.0, 'index': ...",0 days 00:30:00
106306,2023-10-20 21:00:00,2023-10-20T21:30Z,"{'actual': 89.0, 'forecast': 68.0, 'index': 'l...",0 days 00:30:00


In [2]:
# We fill forward in time in cases where there are gaps in the data.
# For forecasts this is completely legitimate
# For realised carbon intensities this is not but sufficient for our case
df.sort_values("from")
df = df.set_index("from")
full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq="30min")
df = df.reindex(full_index, method="ffill")
df = df.rename_axis("from").reset_index()

In [3]:
# Columnise the intensity
cols = pd.json_normalize(df["intensity"])
df = pd.concat([df.drop(columns="intensity"), cols], axis=1)
df

,from,to,actual,forecast,index
0,2017-09-25 23:30:00,2017-09-26T00:00Z,298.0,299.0,moderate
1,2017-09-26 00:00:00,2017-09-26T00:30Z,302.0,293.0,high
2,2017-09-26 00:30:00,2017-09-26T01:00Z,306.0,295.0,high
3,2017-09-26 01:00:00,2017-09-26T01:30Z,306.0,290.0,high
4,2017-09-26 01:30:00,2017-09-26T02:00Z,304.0,289.0,high
...,...,...,...,...,...
147836,2026-03-02 21:30:00,2026-03-02T22:00Z,113.0,126.0,moderate
147837,2026-03-02 22:00:00,2026-03-02T22:30Z,115.0,122.0,moderate
147838,2026-03-02 22:30:00,2026-03-02T23:00Z,120.0,125.0,moderate
147839,2026-03-02 23:00:00,2026-03-02T23:30Z,127.0,125.0,moderate


In [4]:
mask_actual = pd.to_numeric(df["actual"], errors="coerce").notna()
mask_forecast = pd.to_numeric(df["forecast"], errors="coerce").notna()
display(df[~mask_forecast])
display(df[~mask_actual])

,from,to,actual,forecast,index
127967,2025-01-12 23:00:00,2025-01-12T23:30Z,126.0,NaN,moderate
127968,2025-01-12 23:30:00,2025-01-13T00:00Z,126.0,NaN,moderate
127969,2025-01-13 00:00:00,2025-01-13T00:30Z,129.0,NaN,moderate
127970,2025-01-13 00:30:00,2025-01-13T01:00Z,126.0,NaN,moderate
127971,2025-01-13 01:00:00,2025-01-13T01:30Z,123.0,NaN,moderate
127972,2025-01-13 01:30:00,2025-01-13T02:00Z,125.0,NaN,moderate
127973,2025-01-13 02:00:00,2025-01-13T02:30Z,124.0,NaN,moderate
127974,2025-01-13 02:30:00,2025-01-13T03:00Z,125.0,NaN,moderate
127975,2025-01-13 03:00:00,2025-01-13T03:30Z,125.0,NaN,moderate
127976,2025-01-13 03:30:00,2025-01-13T04:00Z,128.0,NaN,moderate


,from,to,actual,forecast,index
20335,2018-11-23 15:00:00,2018-11-23T15:30Z,NaN,416.0,very high
20336,2018-11-23 15:30:00,2018-11-23T16:00Z,NaN,413.0,very high
20337,2018-11-23 16:00:00,2018-11-23T16:30Z,NaN,408.0,very high
20338,2018-11-23 16:30:00,2018-11-23T17:00Z,NaN,403.0,very high
20339,2018-11-23 17:00:00,2018-11-23T17:30Z,NaN,397.0,very high
...,...,...,...,...,...
93532,2023-01-26 13:30:00,2023-01-26T14:00Z,NaN,214.0,high
93533,2023-01-26 14:00:00,2023-01-26T14:30Z,NaN,225.0,high
93534,2023-01-26 14:30:00,2023-01-26T15:00Z,NaN,223.0,high
93535,2023-01-26 15:00:00,2023-01-26T15:30Z,NaN,218.0,high


In [5]:
df = df.ffill()
mask_actual = pd.to_numeric(df["actual"], errors="coerce").notna()
mask_forecast = pd.to_numeric(df["forecast"], errors="coerce").notna()
display(df[~mask_forecast])
display(df[~mask_actual])

,from,to,actual,forecast,index


,from,to,actual,forecast,index


In [6]:
print(type(df["from"][0].time()))

<class 'datetime.time'>


In [ ]:
from datetime import date, time, datetime
from optimiser import Load, Discharge, optimiser, pandas_to_forecast, forecast_charging_stats, realised_charging_stats

# The data is now sufficiently clean for our purposes
# We define a new df: 
# Day | C_naive | C_optimal | C_clairvoyant | S = C_optimal - C_naive | Delta = C_clairvoyant - C_optimal | T = C_clairvoyant - C_naive | eta = S/T |

def times_to_discharges(time_power: list[tuple[time]], date: date) -> np.ndarray[Discharge]:
    discharges = []
    for start, end, power in time_power:
        t_start = np.datetime64(datetime.combine(date, start))
        t_end = np.datetime64(datetime.combine(date, start))
        discharges.append(Discharge(t_start, t_end, power))
    return discharges

#TODO: Define Load
efficiency = 0.95
capacity = 55
charging_rate = 5  
power = 9
discharges = [] 
EV = Load(capacity, charging_rate, discharges, efficiency)
tp1 = (time(7,30), time(8,30), power)
tp2 = (time(17,30), time(18,30), power)
tp = [tp1, tp2]

# TODO: Iterate over DF
def process_df(df: pd.DataFrame, load, time_power):
    # Check we are at start of day
    C_opt = []
    C_clair = []
    dates = []
    for day, chunk in df.groupby(df["from"].dt.floor("D")):
        f = pandas_to_forecast(chunk)
        date = day.date()
        load.discharges = times_to_discharges(time_power, date)
        optimal_charging = optimiser(load, f.time, f.f_carbon_intensity)
        print(optimal_charging)
        clair_charging = optimiser(load, f.time, f.a_carbon_intensity)

        C_clair.append(forecast_charging_stats(clair_charging)["carbon cost"])
        C_opt.append(realised_charging_stats(optimal_charging, f.time, f.a_carbon_intensity)["carbon cost"])
        dates.append(date)
    return pd.DataFrame({"C_opt":C_opt, "C_clair":C_clair, "date": date})

df_small = df[df["from"] > datetime(2026,3,1)]

process_df(df_small, EV, tp)


carbon intensity type: <class 'numpy.float64'>; energy drawn type: <class 'numpy.float64'>
carbon intensity type: <class 'numpy.float64'>; energy drawn type: <class 'numpy.float64'>
13
28
carbon intensity type: <class 'numpy.float64'>; energy drawn type: <class 'numpy.float64'>
carbon intensity type: <class 'numpy.float64'>; energy drawn type: <class 'numpy.float64'>
3
3


,C_opt,C_clair,date
0,0.0,0.0,2026-03-02
1,0.0,0.0,2026-03-02
